# 🏢 [Lab 4] LangChain Middleware 기반 에이전틱 RAG Self-Correction & 2계층 평가 하네스 실무

---

## 🗺️ 엔터프라이즈 미들웨어 하네스 & 2계층 평가 아키텍처

AI 에이전트를 엔터프라이즈 실무에 배포할 때 맞닥뜨리는 **2대 핵심 난제**는 다음과 같습니다:
1. **도구 검색 실패와 환각의 악순환**: 사용자의 모호한 구어체 질문으로 검색이 0건이 되었을 때 맹목적으로 포기하거나, 없는 규정을 모델 가중치로 그럴듯하게 날조(Hallucination)함.
2. **에이전트 블랙박스 문제**: 최종 답변의 옳고 그름만 볼 뿐, 에이전트가 어떤 도구를 거쳤고 쿼리를 어떻게 날렸으며 헛돌지 않았는지 과정(Process)을 감사(Audit)할 수 없음.

본 실습에서는 에이전트의 내부 그래프를 복잡하게 얽어매지 않고, **LangChain 공식 `AgentMiddleware` 표준 스택**을 활용하여 **"도구 레벨 자가수정 ➔ 응답 레벨 환각검증 ➔ 2계층(과정+결과) 종합 평가"**로 이어지는 견고한 하네스 파이프라인을 구축합니다.

```mermaid
flowchart TD
    User["👤 <b>사용자 질문</b>"] --> Agent["🤖 <b>ReAct Core Agent</b><br>(Gemini 3.5 Flash + 3대 RAG 도구)"]

    Agent <--> ToolMW["🛡️ <b>1. 도구 레벨 자가수정 (wrap_tool_call)</b><br>• 0건 공백 실패 감지 (진짜 빈 결과 vs 유효 부정 분리)<br>• QueryRewriter 표준어 정규화 1회 자동 재검색<br>• Anti-Spinning 중복 호출 방지 + 세션 예산 제어"]

    Agent <--> RespMW["🛡️ <b>2. 응답 레벨 자가수정 (after_agent)</b><br>• GroundednessGrader 사실성/환각 심사<br>• 환각 감지 시 [Self-Correction Error] 피드백 주입<br>• 메시지 히스토리 기반 1회 단일 패스 자율 반성"]

    RespMW --> EvalMW["🛡️ <b>3. 2계층 종합 평가 (RAGEvalHarness)</b><br>• 과정(Trajectory 40%) + 결과(RAGAS 60%)<br>• Composite Score 산출 및 세션 감사 로그 적재<br>• 역순 AIMessage 탐색으로 정확한 답변 채점"]

    EvalMW --> Log["📊 <b>session_eval_audit.jsonl 적재</b>"]
```


## 1. 환경 설정 및 베이스라인 에이전트/도구셋 로드

필요한 라이브러리를 로드하고 환경 변수를 확인합니다.
앞서 Lab 1과 Lab 2에서 설계했던 **3대 엔터프라이즈 도구(사내 규정, 한국은행 산업 보고서, 조직도 지식그래프)**를 정의합니다.


In [ ]:
import os
import sys
import time
import json
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents.middleware import AgentMiddleware, wrap_tool_call
from IPython.display import display, Markdown

# 프로젝트 루트 경로 추가 및 .env 로드
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)
load_dotenv(os.path.join(PROJECT_ROOT, ".env"), override=True)

from app.utils.llm import get_llm
from app.utils.context import AgentContext

# 메인 모델 및 심판 모델 셋업
llm = get_llm(model_name="gemini-3.5-flash", temperature=0.0)
judge_llm = get_llm(model_name="gemini-3.5-flash", temperature=0.0)

print(f"✅ 환경 초기화 완료! (LLM: {llm.model_name})")


### 1.1 3대 엔터프라이즈 RAG 도구 정의 (Controlled Test Doubles)

실무 환경을 모사한 3개의 전문 도구를 바인딩합니다:
1. `query_company_policy`: 사내 여비, 복무, 보안 규정 데이터베이스 (존재하지 않는 허위 규정에 대한 명확한 부정 응답 포함)
2. `query_bok_reports`: 한국은행 2024년 분기별 반도체/이차전지/자동차 거시 보고서
3. `query_org_graph`: 사내 부서, 팀장, 결재 승인자, 프로젝트 배정 지식 그래프

> [!NOTE]
> **💡 [하네스 엔지니어링 설계 의도: 왜 제어된 도구(Controlled Tool)를 사용하나요?]**
> * **평가 대상의 순수 격리 (Isolation)**: 4교시의 핵심 학습 목표는 **"LangChain `AgentMiddleware`가 도구 쿼리를 자율 재작성하고, 환각을 감지하여 자가 수정(Self-Correction) 피드백 루프를 완벽히 통제하는가"**입니다.
> * **100% 재현 가능한 벤치마크 (Deterministic Evaluation)**: 실제 Vector DB의 유사도 오차나 네트워크 지연 등 외부 노이즈를 배제하고, 구어체 검색 실패나 허위 규정 환각 시나리오를 결정론적으로 재현하기 위함입니다.
> *(※ 실제 서비스 배포 코드는 `app/agents/corrective_rag_agent.py`에서 실제 `app/database/` 디스크 DB들과 연결되어 동작합니다.)*


In [ ]:
@tool
def query_company_policy(query: str) -> str:
    """
    사내 규정집(출장/여비, 복무, 정보보안, 인사/보수 등)을 정밀 검색합니다.
    사내 정책, 지원금 한도, 출장비, 보안 규칙, 결재 절차에 대한 질문에 사용하세요.
    """
    q_lower = query.lower()
    if any(k in q_lower for k in ["출장", "숙박", "항공", "일비", "학회", "세미나", "학술", "여비"]):
        return (
            "[사내 여비/출장 규정 제14조]\n"
            "1. 해외 출장 숙박비: A등급 지역($250/일), B등급 지역($200/일), C등급 지역($150/일) 실비 상한 지원.\n"
            "2. 항공권 이용 기준: 임원 및 수석 이상은 비즈니스석 탑승, 책임/선임/전임은 이코노미석 탑승이 원칙임.\n"
            "3. 국내 출장 및 학회 참석 일비: 1일당 30,000원 정액 지급 (식비 25,000원 별도 실비 청구 가능)."
        )
    elif ("법인카드" in q_lower or "법인 카드" in q_lower) and any(k in q_lower for k in ["소명", "사유서", "지침", "절차", "사용", "긁"]):
        return (
            "[법인카드 관리 지침 제8조 (주말/휴일 사용 통제)]\n"
            "주말 및 공휴일 법인카드 사용은 원칙적으로 금지됩니다. "
            "불가피한 사업상 목적으로 사용한 경우, 결제일로부터 3영업일 이내에 사내 ERP에 "
            "'주말 사용 사유서'와 증빙 자료(참석자 명단, 업무 회의록 등)를 등록하고 소속 부서장의 사후 결재를 받아야 합니다."
        )
    elif any(k in q_lower for k in ["상여금", "우주", "보수", "특별"]):
        return (
            "[사내 인사/보수 규정 제19조 (상여금 및 복리후생)]\n"
            "사내 보수 규정상 기본급, 일반 정기 성과급, 식대 보조금 기준만 명시되어 있으며 "
            "'우주 항공 개발 특별 연구 상여금' 제도는 사내 규정에 존재하지 않습니다."
        )
    elif "보안" in q_lower or "chatgpt" in q_lower or ("생성형" in q_lower and "ai" in q_lower):
        return (
            "[정보보안 관리 규정 제22조 (생성형 AI 사용 지침)]\n"
            "1. 임직원은 외부 생성형 AI 서비스에 고객 개인정보, 미공개 소스코드, 핵심 재무정보 입력을 엄격히 금지함.\n"
            "2. 위반 행위 적발 시 보안위원회 심의를 거쳐 경고, 감봉, 정직 또는 징계 해고 등 사규에 따른 중징계 처분을 받음."
        )
    else:
        return "사내 규정 데이터베이스에서 해당 키워드와 관련된 유효 규정을 찾지 못했습니다."

@tool
def query_bok_reports(query: str) -> str:
    """
    한국은행 주요 산업(반도체, 이차전지, 자동차, 철강 등) 분기별 경제 동향 보고서를 검색합니다.
    거시 경제 전망, 글로벌 수요/공급망 리스크, 산업별 수출 실적 및 정책 대응에 대한 질문에 사용하세요.
    """
    q_lower = query.lower()
    if "반도체" in q_lower or "hbm" in q_lower or "파운드리" in q_lower:
        return (
            "[한국은행 2024년 4분기 주요 산업 동향 - 반도체 부문]\n"
            "1. HBM 메모리 전망: AI 데이터센터 투자 가속화로 글로벌 HBM 수요는 전년 동기 대비 150% 이상 폭증.\n"
            "2. 주요 공급망 리스크: TSMC CoWoS 등 첨단 어드밴스드 패키징 병목 및 차세대 1b nm 수율 확보 지연.\n"
            "3. 수출 및 가동률: 하반기 반도체 수출은 전년비 28% 증가했으며, 레거시 파운드리 가동률은 2025년 상반기 80% 회복 전망."
        )
    elif "이차전지" in q_lower or "완성차" in q_lower or "ira" in q_lower or "자동차" in q_lower:
        return (
            "[한국은행 2024년 3분기 산업 리포트 - 자동차/배터리]\n"
            "1. IRA 대응 전략: 국내 완성차 업계는 북미 현지 상업용 리스/렌터카 판매 비중을 확대하여 보조금 공백을 방어함.\n"
            "2. 이차전지 합작공장: 북미 주요 OEM(GM, 포드)과의 배터리 합작공장(JV) 조기 완공 및 FTA 체결국 중심 광물 소싱 다변화 추진."
        )
    else:
        return "한국은행 산업 보고서에서 해당 내용에 대한 분석 자료를 찾을 수 없습니다."

@tool
def query_org_graph(query: str) -> str:
    """
    사내 조직도, 부서 구성원, 직속 결재선(상사), 투입된 프로젝트 관계 지식 그래프(Knowledge Graph)를 탐색합니다.
    인물 검색, 조직 관계, 결재 승인자, 프로젝트 참여 인력 확인에 사용하세요.
    """
    q_lower = query.lower()
    if "김철수" in q_lower or "클라우드운영팀" in q_lower:
        return (
            "[사내 지식 그래프 Subgraph]\n"
            "• 엔티티: 김철수 (직급: 팀장, 소속: 클라우드사업본부 클라우드운영팀)\n"
            "• 직속 상사(결재권자): 최동식 본부장 (직급: 상무, 소속: 클라우드사업본부)\n"
            "• 참여 프로젝트: '엔터프라이즈 하이브리드 클라우드 전환(Cloud-Next)' (역할: 총괄 PM)"
        )
    elif "박영희" in q_lower or "ai솔루션" in q_lower:
        return (
            "[사내 지식 그래프 Subgraph]\n"
            "• 엔티티: 박영희 (직급: 수석, 소속: AI솔루션본부 AI솔루션팀)\n"
            "• 직속 상사(결재권자): 이민호 본부장 (직급: 상무, 소속: AI솔루션본부)\n"
            "• 참여 프로젝트: '대규모 언어모델 에이전트 플랫폼 구축' (역할: 테크 리드)"
        )
    elif "sf-2025" in q_lower or "스마트팩토리" in q_lower:
        return (
            "[사내 지식 그래프 Subgraph]\n"
            "• 프로젝트: 스마트팩토리 구축 프로젝트 (SF-2025)\n"
            "• 총괄 책임자: 정우성 수석 (소속: 제조DX팀, 직급: 부장급 수석)\n"
            "• 투입 인력 소속 부서: 제조DX팀(5명), 클라우드운영팀(2명), 보안엔지니어링팀(2명)"
        )
    else:
        return "지식 그래프에서 해당 엔티티 및 관계를 조회할 수 없습니다."

tools = [query_company_policy, query_bok_reports, query_org_graph]
print(f"✅ 3대 엔터프라이즈 도구 바인딩 완료: {[t.name for t in tools]}")


## 2. [문제 제기] Naive ReAct 에이전트의 2대 취약점

미들웨어 하네스가 없는 순수 Naive ReAct 에이전트를 먼저 생성하여 실행해 봅니다.

```
[Naive ReAct의 한계]
1. 모호한 비격식 질문 ➔ 키워드 매칭 실패 ➔ 맹목적 포기 (Retry Waste)
2. 사내에 없는 허위 규정 질문 ➔ 모델 가중치(Parametric Memory)로 그럴듯하게 날조 (Hallucination)
```


In [ ]:
# 순수 Naive ReAct 에이전트 (미들웨어 없음)
naive_agent = create_agent(
    model=llm,
    tools=tools,
    checkpointer=MemorySaver(),
    middleware=[],
    context_schema=AgentContext
)

print("🤖 베이스라인 Naive ReAct Agent 생성 완료!")


### 2.1 [테스트 1] 구어체/모호한 쿼리로 인한 검색 실패 사례
사용자가 *"그 뭐냐 주말에 법카 긁었을 때 소명하는 거 어떻게 해야 됨?"* 이라고 질문했을 때, Naive ReAct는 원본 구어체('법카 긁었을 때')를 그대로 검색하여 공백 결과를 받고 포기하거나 부정확한 답변을 냅니다.


In [ ]:
cfg_t1 = {"configurable": {"thread_id": "naive_test_1"}}
q1 = "그 뭐냐 주말에 법카 긁었을 때 소명하는 거 어떻게 해야 됨?"
resp_t1 = naive_agent.invoke({"messages": [{"role": "user", "content": q1}]}, config=cfg_t1)
print(f"❓ [질문]: {q1}\n")
print(f"🤖 [Naive 답변]:\n{resp_t1['messages'][-1].content}")


### 2.2 [테스트 2] 사내에 없는 허위 규정에 대한 환각(Hallucination) 유도 사례
사내에 존재하지 않는 *"2026년 신규 도입된 우주 항공 개발 특별 연구 상여금 지급 기준과 금액"*을 질문했을 때, Naive Agent는 '규정에 없다'고 선을 긋지 못하고 그럴듯한 가상의 수치를 지어낼 위험이 있습니다.


In [ ]:
cfg_t2 = {"configurable": {"thread_id": "naive_test_2"}}
q2 = "2026년도 신규 도입된 '우주 항공 개발 특별 연구 상여금'의 지급 대상과 1인당 최대 지급 금액은 얼마인가요?"
resp_t2 = naive_agent.invoke({"messages": [{"role": "user", "content": q2}]}, config=cfg_t2)
print(f"❓ [질문]: {q2}\n")
print(f"🤖 [Naive 답변]:\n{resp_t2['messages'][-1].content}")


## 3. [Part 1] 도구 레벨 자가 수정 미들웨어 (`RAGToolCorrectionMiddleware`)

### 📌 왜 도구 레벨 자가 수정이 필요한가요?
엔터프라이즈 실무에서 사용자는 *"그 뭐냐 주말에 법카 긁은 거 소명..."*처럼 **일상 구어체, 축약어, 은어**로 질문합니다.  
하지만 사내 DB는 표준 비즈니스 용어(`"법인카드 주말 사용 소명 절차"`)로 인덱싱되어 있어 **첫 검색에서 0건(공백 결과)**이 반환됩니다.
* **Naive ReAct의 문제**: *"결과가 없네?"* 하고 맹목적으로 포기하거나, 똑같은 쿼리로 계속 도구를 불러 **무한 루프(Spinning)**에 빠집니다.
* **미들웨어의 해결책**: 도구 실행 중간에 개입하여 공백 결과를 감지하고, **가벼운 `QueryRewriter` LLM을 통해 표준 비즈니스 키워드로 정규화한 후 딱 1회 자동으로 재검색**을 수행합니다.

```mermaid
flowchart TD
    A["🤖 ReAct Core Agent<br>도구 호출: query='주말에 법카 긁은 거'"] --> B["🛡️ wrap_tool_call 가로채기"]
    B --> C["1. Anti-Spinning 중복 호출 검사"]
    C --> D["2. 1차 도구 실행: handler(request)"]
    D --> E{"3. 결과 판별<br>(_should_retry)"}
    E -- "유효 문서 획득 또는<br>유효 부정('규정 미존재')" --> H["결과 반환 & Trajectory 로깅"]
    E -- "진짜 0건 공백 실패" --> F["🔄 QueryRewriter LLM 정규화<br>'주말 법인카드 사용 소명 절차'"]
    F --> G["도구 인자 교체 후 1회 자동 재실행<br>(세션 예산 max_session_rewrites 차감)"]
    G --> H
```

### 🔍 핵심 구현 원리
1. **`@wrap_tool_call` 훅**: LangChain `AgentMiddleware`의 표준 데코레이터로, 모든 도구의 실행 전후를 가로채어 실행 시간 측정, 인자 변조, 조건부 재실행을 완벽히 제어합니다.
2. **`RewrittenQuery` (Pydantic 구조화 출력)**: LLM에게 단순 텍스트가 아닌 정규화된 쿼리(`rewritten_query`)와 확장 키워드(`expanded_keywords`)를 JSON 스키마로 강제 생성받습니다.
3. **Anti-Spinning Cache & 세션 예산 제어**: 동일한 도구에 동일한 인자로 헛도는 호출을 감지하고, 세션 단위 총 재작성 횟수(`max_session_rewrites=3`)를 제한하여 무한 루프를 방지합니다.
4. **유효 부정 응답 보호**: `"사내 규정에 존재하지 않습니다"`와 같은 유효한 부정 응답은 공백 결과로 오분류하지 않고 즉시 통과시킵니다.

> [!TIP]
> **💡 프로덕션 모듈 분리**:
> 실제 코드는 `app/middleware/rag_tool_correction.py`에 완성되어 있으므로, 노트북에서는 깔끔하게 **import**하여 사용합니다.


In [ ]:
# app/middleware 패키지에서 프로덕션 미들웨어를 import합니다.
from app.middleware import (
    RAGToolCorrectionMiddleware,
    RewrittenQuery,
)

print("✅ RAGToolCorrectionMiddleware 로드 완료!")
print(f"   📦 모듈 경로: {RAGToolCorrectionMiddleware.__module__}")
print(f"   🔧 주요 훅: wrap_tool_call (도구 실행 가로채기 → 빈 결과 시 쿼리 재작성 → 1회 자동 재시도)")
print(f"   🛡️ 안전장치: 세션 단위 총 재작성 예산(max_session_rewrites) + 유효 부정 응답 보호")


## 4. [Part 2] 응답 레벨 환각 검증 & 자율 반성 미들웨어 (`RAGSelfCorrectionMiddleware`)

### 📌 왜 응답 레벨 환각 검증이 필요한가요?
RAG 시스템에서 검색된 문서(Context)에 원하는 답이 없거나 사내에 존재하지 않는 허위 규정을 물었을 때, LLM은 종종 **학습된 가중치(Parametric Memory)에 의존하여 그럴듯한 거짓말(환각)**을 답변으로 내놓습니다.  
이는 금융, 인사, 법무 RAG 시스템에서 **심각한 규정 위반 및 감사 사고**를 유발합니다.
* **Naive ReAct의 문제**: 프롬프트에 *"문서에 없으면 답하지 마세요"*라고 적어도, 유도성 질문에 쉽게 넘어가 허위 답변을 생성함.
* **미들웨어의 해결책**: 에이전트가 답변을 완성했을 때 사용자에게 내보내기 직전 `after_agent`로 가로채어, **`GroundednessGrader` 심판관이 검색 문서와 답변을 대조 채점**합니다. 사실성이 미달되면 즉시 차단(Blocking)하고 교정 피드백 메시지를 주입하여 에이전트가 스스로 정정(Self-Correction)하도록 만듭니다.

```mermaid
flowchart TD
    A["🤖 ReAct Core Agent<br>최종 AIMessage 답변 완성"] --> B["🛡️ after_agent 라이프사이클 가로채기"]
    B --> C["1. Context & 마지막 AIMessage 답변 추출"]
    C --> D["2. GroundednessGrader LLM 심판<br>(is_grounded, has_hallucination, score)"]
    D --> E{"3. 기준 점수 판정<br>Score ≥ 0.70 & 무환각?"}
    E -- "통과 (합격)" --> F["✅ 승인: 사용자에게 최종 답변 전달"]
    E -- "미달 / 환각 감지" --> G{"자가 수정 시도 < max_retries(1회)?"}
    G -- "예 (1회 한정)" --> H["🛑 [Self-Correction Blocking Error]<br>피드백 메시지 주입 후 재트리거"]
    H --> I["🔄 에이전트 자가 반성 및 정정 답변 생성"]
    I --> B
    G -- "아니오 (예산 소진)" --> F
```

### 🔍 핵심 구현 원리
1. **`after_agent` 라이프사이클 훅**: 에이전트 루프가 끝나기 직전 최종 대화 상태(`state["messages"]`)를 가로챕니다.
2. **상태 주입을 통한 자가 수정 격발**: 미들웨어가 `return {"messages": [HumanMessage(feedback_msg)]}`를 반환하면, LangGraph 에이전트는 이를 새로운 사용자 피드백으로 인지하여 **내부 추론 루프를 한 번 더 돌며 답변을 스스로 정정**합니다.
3. **메시지 히스토리 기반 단일 패스(One-Shot) 통제**: 외부 dict 대신 대화 메시지 히스토리 내 `🛑 [Self-Correction` 프리픽스를 직접 카운트하여, 멀티스레드 환경에서도 상태 꼬임 없이 최대 1회의 깔끔한 정정을 보장합니다.

> [!TIP]
> **💡 프로덕션 모듈 분리**:
> 실제 코드는 `app/middleware/rag_self_correction.py`에 완성되어 있으므로, 노트북에서는 깔끔하게 **import**하여 사용합니다.


In [ ]:
# app/middleware 패키지에서 프로덕션 미들웨어를 import합니다.
from app.middleware import (
    RAGSelfCorrectionMiddleware,
    GroundednessEvaluation,
    CORRECTION_PREFIX,
)

print("✅ RAGSelfCorrectionMiddleware 로드 완료!")
print(f"   📦 모듈 경로: {RAGSelfCorrectionMiddleware.__module__}")
print(f"   🔧 주요 훅: after_agent (에이전트 답변 후 → Groundedness Judge → 환각 시 피드백 주입)")
print(f"   🛡️ 안전장치: 메시지 히스토리 기반 재시도 카운트 (스레드 안전) + one-shot correction")
print(f"   📌 피드백 식별자: '{CORRECTION_PREFIX}...'")


## 5. [Part 3] 과정(Trajectory) & 결과(RAGAS) 2계층 평가 하네스 (`RAGEvalHarnessMiddleware`)

### 📌 왜 '2계층(2-Tier)' 평가가 필수적인가요?
기존 RAG 평가는 **결과(답변 정확도)**만 봤습니다. 하지만 다단계 도구를 자율 실행하는 에이전틱 RAG에서는 **치명적인 맹점**이 존재합니다:
* **상황 예시**: 쉬운 규정 질문인데, 에이전트가 엉뚱한 도구를 5번 호출하고 쿼리를 헛돌리며 **토큰을 낭비하고 15초 뒤에 겨우 정답**을 맞혔다면?
  * **기존 RAGAS 평가**: *"최종 답은 맞았으니 100점!"* ➔ **비용 낭비와 느린 속도를 전혀 감지 못함 (❌)**
  * **2계층 평가 하네스**: *"결과는 100점이지만, 과정(Step Economy)이 30점 감점되어 종합 58점!"* ➔ **에이전트의 비효율성을 정확히 적발 (⭕)**

```mermaid
flowchart TD
    N1["<b>1. 라이프사이클 훅 (Lifecycle Hooks)</b><br>────────────────────────────────────<br>• before_agent: 세션 타이머 & 궤적 버퍼 초기화<br>• wrap_tool_call: 실시간 도구명, 인자, 지연시간(Latency) 캡처<br>• after_agent: 2계층 종합 채점 및 JSONL 로그 적재"]

    N2["<b>2. 과정 평가 (Trajectory - 40%)</b><br>────────────────────────────────────<br>• 도구 선택 적절성 (tool_selection_accuracy)<br>• 검색 쿼리 품질 (argument_quality_score)<br>• 궤적 경제성 / 낭비 방지 (step_economy_score)<br>• 관찰값 인용도 (observation_grounding_score)"]

    N3["<b>3. 결과 평가 (RAGAS Outcome - 60%)</b><br>────────────────────────────────────<br>• 충실도 / 무환각 (Faithfulness)<br>• 답변 관련성 (Answer Relevance)<br>• 컨텍스트 정밀도 (Context Precision)<br>• 컨텍스트 재현율 (Context Recall)"]

    N4["🏆 <b>Composite Score 산출</b><br>Formula = (Traj × 0.40) + (Outcome × 0.60)"]

    N5["💾 <b>session_eval_audit.jsonl 영구 적재</b>"]

    N1 --> N2
    N1 --> N3
    N2 --> N4
    N3 --> N4
    N4 --> N5
```

> [!IMPORTANT]
> **🏭 [실무 엔지니어링 Q&A: 실서비스에서도 매번 동기(Sync)로 평가하나요? - 지연시간 및 비용 제로화 아키텍처]**
> * **사용자 대기시간(Latency) 이슈**: 유저가 실시간 채팅 중일 때 매 턴마다 심판 LLM을 동기로 2회씩 호출하면 응답이 3~5초 지연됩니다.
> * **실무 운영 아키텍처 (Production Architecture)**:
>   1. **비동기 백그라운드 워커 (Async Background Queue)**: 에이전트는 사용자에게 답변을 **즉시 스트리밍으로 전송**하고 세션을 끝냅니다. 대화 궤적 로그는 `FastAPI BackgroundTasks`나 `Redis Queue(Celery/Kafka)`로 비동기 전달되어, 백그라운드 워커가 조용히 채점합니다. (➔ **사용자 지연시간 = 0ms**)
>   2. **트래픽 표본 샘플링 (Traffic Sampling)**: LLM 평가 비용 절감을 위해 전체 트래픽의 **5%~10%만 무작위 표본 추출**하여 평가합니다.
>   3. **CI/CD 오프라인 게이트웨이 (Quality Gate)**: 에이전트 코드/프롬프트 배포 전 GitHub Actions에서 100개 골든 케이스를 자동 채점하여 `Composite Score ≥ 0.85`일 때만 배포를 승인합니다.
> * **본 실습(Lab 4)에서 인라인으로 실행하는 이유**: 교육생 여러분이 복잡한 큐 인프라 없이도, Naive 대비 미들웨어 하네스의 품질 개선 폭을 그 자리에서 즉시 성적표와 차트로 체감하기 위함입니다.

```mermaid
flowchart TD
    User["👤 <b>사용자 질문</b>"] --> Agent["🤖 <b>RAG Agent</b><br>(Tool & Self Correction)"]
    Agent --> FastResp["⚡ <b>실시간 스트리밍 답변 완료</b><br>(사용자에게 즉시 전달)"]
    
    Agent -.->|"비동기 덤프 (0ms Latency)"| Queue["📥 <b>비동기 메시지 큐</b><br>(Redis / Celery / Kafka)"]
    
    Queue --> Sampler{"🎲 <b>트래픽 표본 샘플링</b><br>(전체 트래픽 5~10%)"}
    Sampler -- "선정된 세션" --> Judge["⚖️ <b>LLM-as-a-Judge 채점</b><br>(RAGEvalHarness 2-Tier)"]
    Judge --> DB["📊 <b>모니터링 대시보드 적재</b><br>(LangSmith / Arize / Datadog)"]
```

### 🔍 2계층 평가 수식
1. **과정 점수 (Trajectory Score 40%)**: 도구선택 정확도 + 쿼리 품질 + 궤적 경제성 + 관측값 인용도의 산술 평균
2. **결과 점수 (RAGAS Outcome Score 60%)**: Faithfulness + Answer Relevance + Context Precision + Context Recall의 산술 평균
3. **종합 점수 (Composite Score)**:
$$\text{Composite Score} = (\text{Trajectory Score} \times 0.40) + (\text{Outcome Score} \times 0.60)$$


In [ ]:
# app/middleware 패키지에서 프로덕션 미들웨어를 import합니다.
from app.middleware import (
    RAGEvalHarnessMiddleware,
    TrajectoryEvaluation,
    OutcomeEvaluation,
    ComprehensiveEvalResult,
    extract_last_ai_answer,
    extract_user_query,
    extract_tool_contexts,
)

print("✅ RAGEvalHarnessMiddleware 로드 완료!")
print(f"   📦 모듈 경로: {RAGEvalHarnessMiddleware.__module__}")
print(f"   🔧 주요 훅: before_agent → wrap_tool_call → after_agent (전 라이프사이클)")
print(f"   📊 평가 체계: Trajectory(40%) + RAGAS Outcome(60%) = Composite Score")
print(f"   🛡️ 핵심 수정: extract_last_ai_answer() — 역순 AIMessage 탐색으로 정확한 답변 추출")


## 6. [Part 4] 실전 벤치마크 대결: Naive Agent vs Middleware-Harnessed Agent

이제 10개의 엔터프라이즈 골든 데이터셋(`data/eval/golden_eval_dataset.json`)을 대상으로:
1. **Naive ReAct Agent (미들웨어 없음)**: 도구 검색 실패 시 쿼리 교정이나 응답 환각 검증 장치가 없는 순수 에이전트
2. **Middleware-Harnessed RAG Agent (3대 미들웨어 탑재)**: `RAGToolCorrection` + `RAGSelfCorrection` + `RAGEvalHarness`
두 시스템의 성능을 일괄 배치 실행하고 정량 비교합니다.


In [ ]:
# 1. 골든 데이터셋 로드
eval_dataset_path = os.path.join(PROJECT_ROOT, "data", "eval", "golden_eval_dataset.json")
with open(eval_dataset_path, "r", encoding="utf-8") as f:
    golden_dataset = json.load(f)

print(f"📂 골든 평가 데이터셋 {len(golden_dataset)}개 문항 로드 완료!")
for item in golden_dataset[:3]:
    print(f"  • [{item['id']}] {item['question']}")


### 6.1 Middleware-Harnessed RAG 에이전트 조립


In [ ]:
# 미들웨어 인스턴스화
tool_correction_mw = RAGToolCorrectionMiddleware(
    llm=llm,
    enable_auto_retry=True,
    max_session_rewrites=3,  # 세션 단위 총 재작성 예산
    verbose=True
)
self_correction_mw = RAGSelfCorrectionMiddleware(
    judge_llm=judge_llm,
    min_groundedness_score=0.70,
    max_retries=1,           # one-shot correction (무한 루프 방지)
    verbose=True
)
eval_harness_mw = RAGEvalHarnessMiddleware(
    judge_llm=judge_llm,
    verbose=True
)

# Harnessed Agent 구축
harnessed_agent = create_agent(
    model=llm,
    tools=tools,
    checkpointer=MemorySaver(),
    middleware=[tool_correction_mw, self_correction_mw, eval_harness_mw],
    context_schema=AgentContext
)

print("🛡️ Middleware-Harnessed Agent 조립 완료!")
print(f"   미들웨어 체인: {[type(m).__name__ for m in [tool_correction_mw, self_correction_mw, eval_harness_mw]]}")


### 6.2 10개 문항 고속 병렬(Concurrent) 벤치마크 실행 및 트레이스 수집

`app/utils/benchmark.py`의 `run_parallel_benchmark()` 함수를 호출하여, **5개 워커 스레드로 10개 골든 케이스를 병렬 배치 평가**합니다.  
순차 실행 대비 평가 소요 시간이 대폭 단축되며, 각 스레드별로 격리된 미들웨어 인스턴스를 통해 과정(Trajectory)과 결과(RAGAS)를 동시 채점합니다.


In [ ]:
# app/utils/benchmark 패키지에서 고속 병렬 벤치마크 엔진을 import합니다.
from app.utils.benchmark import run_parallel_benchmark

# 🚀 10개 골든 케이스 고속 병렬(Concurrent) 벤치마크 실행 (5개 워커 스레드)
df_results = run_parallel_benchmark(
    golden_dataset=golden_dataset,
    tools=tools,
    llm=llm,
    judge_llm=judge_llm,
    max_workers=5,
    naive_agent=naive_agent,
    verbose=True,
)


### 6.3 정량 평가 종합 데이터프레임 및 통계 분석


In [ ]:
# 요약 통계 테이블 출력
display_cols = ["ID", "Domain", "Naive_Faithfulness", "Harness_Faithfulness", "Naive_Composite", "Harness_Composite", "Naive_Latency_ms", "Harness_Latency_ms"]
print("📊 [벤치마크 결과 상세표]")
print(df_results[display_cols].to_markdown(index=False))

# 평균 지표 비교
summary_stats = pd.DataFrame({
    "지표 (Metric)": ["Faithfulness (충실도)", "Answer Relevance (관련성)", "Trajectory Score (과정 점수)", "Composite Total (종합 점수)", "Avg Latency (ms)"],
    "Naive ReAct": [
        df_results["Naive_Faithfulness"].mean(),
        df_results["Naive_Relevance"].mean(),
        df_results["Naive_TrajScore"].mean(),
        df_results["Naive_Composite"].mean(),
        df_results["Naive_Latency_ms"].mean(),
    ],
    "Middleware Harnessed": [
        df_results["Harness_Faithfulness"].mean(),
        df_results["Harness_Relevance"].mean(),
        df_results["Harness_TrajScore"].mean(),
        df_results["Harness_Composite"].mean(),
        df_results["Harness_Latency_ms"].mean(),
    ]
})

print("\n🏆 [최종 종합 평균 성적표]")
print(summary_stats.to_markdown(index=False))

# 개선폭 출력
improvement = df_results['Harness_Composite'].mean() - df_results['Naive_Composite'].mean()
print(f"\n{'📈 개선' if improvement > 0 else '📉 저하'}: Composite Score 평균 {improvement:+.3f} ({'Harness 우위' if improvement > 0 else 'Naive 우위'})")


### 6.4 📊 정량 비교 시각화 차트 생성 (Matplotlib)


In [ ]:
plt.figure(figsize=(12, 5))

# 1. Comparison of 4 Quality & Trajectory Metrics (Bar Chart)
plt.subplot(1, 2, 1)
metrics = ["Faithfulness", "Relevance", "Trajectory", "Composite"]
naive_scores = [
    df_results["Naive_Faithfulness"].mean(),
    df_results["Naive_Relevance"].mean(),
    df_results["Naive_TrajScore"].mean(),
    df_results["Naive_Composite"].mean()
]
harness_scores = [
    df_results["Harness_Faithfulness"].mean(),
    df_results["Harness_Relevance"].mean(),
    df_results["Harness_TrajScore"].mean(),
    df_results["Harness_Composite"].mean()
]

x = range(len(metrics))
width = 0.35
plt.bar([i - width/2 for i in x], naive_scores, width=width, label="Naive ReAct", color="#ff7675")
plt.bar([i + width/2 for i in x], harness_scores, width=width, label="Middleware Harnessed", color="#00b894")
plt.ylabel("Score (0.0 ~ 1.0)", fontsize=11)
plt.title("RAG Quality & Trajectory Metric Comparison", fontsize=12, fontweight="bold")
plt.xticks(x, metrics, fontsize=10)
plt.ylim(0, 1.1)
plt.legend(loc="upper right")
plt.grid(axis="y", linestyle="--", alpha=0.7)

# 2. Composite Score Trend Across 10 Golden Cases (Line Plot)
plt.subplot(1, 2, 2)
plt.plot(df_results["ID"], df_results["Naive_Composite"], marker="o", linewidth=2, color="#d63031", label="Naive ReAct")
plt.plot(df_results["ID"], df_results["Harness_Composite"], marker="s", linewidth=2, color="#0984e3", label="Middleware Harnessed")
plt.xlabel("Golden Case ID", fontsize=11)
plt.ylabel("Composite Score (0.0 ~ 1.0)", fontsize=11)
plt.title("10 Golden Benchmark Cases: Composite Score Trend", fontsize=12, fontweight="bold")
plt.xticks(rotation=45, fontsize=9)
plt.ylim(0, 1.1)
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
chart_path = os.path.join(PROJECT_ROOT, "artifacts", "rag_benchmark_comparison.png")
os.makedirs(os.path.dirname(chart_path), exist_ok=True)
plt.savefig(chart_path, dpi=200)
plt.show()

print(f"✅ Comparison chart saved successfully: {chart_path}")


## 7. 🎓 엔터프라이즈 에이전틱 RAG 실무 핵심 요약 (Key Engineering Takeaways)

---

### 🏆 3대 미들웨어의 실무 배포 아키텍처 비교 요약

실무에서는 성능(지연시간)과 안전성(무환각)의 균형을 위해 각 미들웨어를 다음과 같이 배치합니다:

| 미들웨어 (Middleware) | 프로덕션 운영 방식 | 사용자 응답 영향 (Latency) | 주요 역할 및 배포 가이드라인 |
| :--- | :--- | :---: | :--- |
| **`RAGToolCorrection`** | **인라인 (실시간 동기)** | ~0.2초 (경량 LLM) | 0건 검색 실패 시 즉시 유효 문서를 회수하기 위해 **필수 활성화** |
| **`RAGSelfCorrection`** | **인라인 (선택적 가드레일)** | ~0.5초 | 금융, 인사, 법무 등 **환각 0%가 절대적인 미션 크리티컬 도메인**에 활성화 |
| **`RAGEvalHarness`** | **비동기 백그라운드 워커 / CI/CD** | **0초 (영향 없음)** | 전체 트래픽 5~10% 샘플링 감사 또는 **배포 전 GitHub Actions 품질 게이트** |

---

### 🚀 프로덕션 배포 및 웹 UI 연동
완성된 에이전트 코드는 `app/agents/corrective_rag_agent.py`에 등록되어 있으며, 다음 명령어로 Streamlit 웹 UI에서 즉시 대화형으로 테스트할 수 있습니다:

```bash
# Streamlit 웹 UI 가동
streamlit run app/ui.py
```
* 웹 브라우저(`http://localhost:8501`)에서 **"corrective_rag_agent"**를 선택하여 미들웨어 하네스의 실시간 자가 수정 궤적을 확인해 보세요!
